# Wafer Map Defect Classification: Part 4, Fair Final Comparison, 3 Seeds and Demo
**Before you run:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → Run all.

At the upload button, select **results.zip** (Part 1) and **results3.zip** (Part 3) together (hold Ctrl). results.zip is required. results3.zip lets this notebook reuse your seed-42 CNN and Hybrid runs instead of training them again.

What this notebook does, all with the Part 3 recipe (augmentation, class weights, patience 10):
1. Trains the **Baseline CNN** and the **Hybrid** on 2 more seeds (43, 44) to get mean ± std
2. Retrains **ViT** and **ResNet-18** with the same recipe, so every model in the table is trained the same way
3. Builds the **final paper table** and a per-class F1 table
4. Launches an optional **Gradio demo**: pick a defect type, see the wafer, the prediction and the Grad-CAM heatmap

Run time is about 80–100 minutes on a T4. Every model's result is saved the moment it finishes. If Colab disconnects, run the **partial download** cell in section 3 first.

In [ ]:
import os, time, zipfile, shutil, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf, keras
from keras import layers as L
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

from google.colab import files
up = files.upload()                        # select results.zip and results3.zip
for f in up: zipfile.ZipFile(f).extractall('results3_old' if 'results3' in f else '.')

CLASSES = ['Center', 'Donut', 'Edge-Loc', 'Edge-Ring', 'Loc', 'Near-full', 'Random', 'Scratch', 'none']
IMG, NC, BS = 64, 9, 128
os.makedirs('results4', exist_ok=True)

d = np.load('results/splits_64.npz')
Xtr_i, y_tr = d['X_tr'], d['y_tr']
onehot = lambda a: np.eye(3, dtype=np.float32)[a]
X_va, y_va = onehot(d['X_va']), d['y_va']
X_te, y_te = onehot(d['X_te']), d['y_te']
cw = compute_class_weight('balanced', classes=np.arange(NC), y=y_tr)
class_weight = {i: float(np.sqrt(w)) for i, w in enumerate(cw)}
print('train', Xtr_i.shape, 'test', X_te.shape, '| GPU:', tf.config.list_physical_devices('GPU'))

## 1. Models and training recipe (same as Part 3)

In [ ]:
@keras.saving.register_keras_serializable()
class PositionEmbedding(L.Layer):
    def build(self, input_shape):
        self.pos = self.add_weight(name='pos', shape=(1, input_shape[1], input_shape[2]), initializer='random_normal')
    def call(self, x):
        return x + self.pos

def transformer_block(x, dim, heads=4, mlp=256, drop=0.1, name=None):
    h = L.LayerNormalization(epsilon=1e-6)(x)
    h = L.MultiHeadAttention(num_heads=heads, key_dim=dim // heads, dropout=drop, name=name)(h, h)
    x = L.Add()([x, h])
    h = L.LayerNormalization(epsilon=1e-6)(x)
    h = L.Dense(mlp, activation='gelu')(h); h = L.Dropout(drop)(h)
    h = L.Dense(dim)(h); h = L.Dropout(drop)(h)
    return L.Add()([x, h])

def conv_bn(x, f, k=3, s=1, act=True, name=None):
    x = L.Conv2D(f, k, strides=s, padding='same', use_bias=False, name=name)(x)
    x = L.BatchNormalization()(x)
    return L.ReLU()(x) if act else x

def build_cnn():                                   # same as Part 1 baseline
    inp = L.Input((IMG, IMG, 3)); x = inp
    for f in [32, 64, 128, 256]:
        x = L.Conv2D(f, 3, padding='same', activation='relu')(x)
        x = L.BatchNormalization()(x)
        x = L.Conv2D(f, 3, padding='same', activation='relu')(x)
        x = L.MaxPooling2D()(x); x = L.Dropout(0.25)(x)
    x = L.GlobalAveragePooling2D()(x)
    x = L.Dense(128, activation='relu')(x); x = L.Dropout(0.4)(x)
    return keras.Model(inp, L.Dense(NC, activation='softmax')(x), name='Baseline_CNN_v2')

def build_hybrid(dim=128, depth=2):                # same as Part 2 hybrid
    inp = L.Input((IMG, IMG, 3))
    x = conv_bn(inp, 32); x = conv_bn(x, 32); x = L.MaxPooling2D()(x)
    x = conv_bn(x, 64); x = conv_bn(x, 64); x = L.MaxPooling2D()(x)
    x = conv_bn(x, dim); x = conv_bn(x, dim, name='last_conv'); x = L.MaxPooling2D()(x)
    x = L.Reshape((-1, dim))(x)
    x = PositionEmbedding()(x)
    for i in range(depth): x = transformer_block(x, dim, name=f'mhsa_{i}')
    x = L.LayerNormalization(epsilon=1e-6)(x)
    x = L.GlobalAveragePooling1D()(x); x = L.Dropout(0.3)(x)
    return keras.Model(inp, L.Dense(NC, activation='softmax')(x), name='Hybrid_CNN_Transformer_v2')

def build_resnet18():
    def block(x, f, s):
        sc = x
        x = conv_bn(x, f, 3, s); x = conv_bn(x, f, 3, 1, act=False)
        if s != 1 or sc.shape[-1] != f: sc = conv_bn(sc, f, 1, s, act=False)
        return L.ReLU()(L.Add()([x, sc]))
    inp = L.Input((IMG, IMG, 3)); x = conv_bn(inp, 64, 3, 1)
    for f, s in [(64, 1), (128, 2), (256, 2), (512, 2)]:
        x = block(x, f, s); x = block(x, f, 1)
    x = L.GlobalAveragePooling2D()(x); x = L.Dropout(0.3)(x)
    return keras.Model(inp, L.Dense(NC, activation='softmax')(x), name='ResNet18_v2')

def build_vit(dim=128, depth=4):
    inp = L.Input((IMG, IMG, 3))
    x = L.Conv2D(dim, 8, strides=8)(inp); x = L.Reshape((-1, dim))(x); x = PositionEmbedding()(x)
    for i in range(depth): x = transformer_block(x, dim)
    x = L.LayerNormalization(epsilon=1e-6)(x)
    x = L.GlobalAveragePooling1D()(x); x = L.Dropout(0.3)(x)
    return keras.Model(inp, L.Dense(NC, activation='softmax')(x), name='ViT_v2')

def make_ds(seed):
    def aug(x, y):
        x = tf.image.rot90(x[..., None], k=tf.random.uniform([], 0, 4, dtype=tf.int32))
        x = tf.cond(tf.random.uniform([]) < 0.5, lambda: tf.image.flip_left_right(x), lambda: x)
        return tf.one_hot(tf.cast(x[..., 0], tf.int32), 3), y
    return (tf.data.Dataset.from_tensor_slices((Xtr_i, y_tr)).shuffle(len(y_tr), seed=seed, reshuffle_each_iteration=True)
            .map(aug, num_parallel_calls=tf.data.AUTOTUNE).batch(BS).prefetch(tf.data.AUTOTUNE))

In [ ]:
LOG = 'results4/all_runs.csv'
COLS = ['model', 'seed', 'params', 'size_MB', 'test_accuracy', 'macro_F1', 'inference_ms_per_wafer', 'train_time_min'] + [f'F1_{c}' for c in CLASSES]
def log_row(r): pd.DataFrame([r]).reindex(columns=COLS).to_csv(LOG, mode='a', header=not os.path.exists(LOG), index=False)
def run(builder, seed, epochs=60):
    keras.backend.clear_session(); keras.utils.set_random_seed(seed)
    model = builder()
    model.compile(optimizer=keras.optimizers.AdamW(1e-3, weight_decay=1e-4),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    cb = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
          keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5)]
    print(f'\n===== {model.name}  seed {seed} =====')
    t0 = time.time()
    model.fit(make_ds(seed), validation_data=(X_va, y_va), epochs=epochs, class_weight=class_weight, callbacks=cb, verbose=2)
    tt = (time.time() - t0) / 60
    yp = model.predict(X_te, batch_size=256, verbose=0).argmax(1)
    fn = f'results4/{model.name}_s{seed}.keras'; model.save(fn)
    _ = model.predict(X_te[:64], batch_size=1, verbose=0)
    t0 = time.time(); _ = model.predict(X_te[:500], batch_size=1, verbose=0)
    row = {'model': model.name, 'seed': seed, 'params': int(model.count_params()), 'size_MB': round(os.path.getsize(fn) / 1e6, 2),
           'test_accuracy': round(accuracy_score(y_te, yp), 4), 'macro_F1': round(f1_score(y_te, yp, average='macro'), 4),
           'inference_ms_per_wafer': round((time.time() - t0) / 500 * 1000, 3), 'train_time_min': round(tt, 1)}
    row.update({f'F1_{c}': round(v, 4) for c, v in zip(CLASSES, f1_score(y_te, yp, average=None, labels=range(NC)))})
    log_row(row)   # saved immediately
    print({k: row[k] for k in ['model', 'seed', 'test_accuracy', 'macro_F1']})
    if 'ResNet' in model.name and os.path.exists(fn): os.remove(fn)   # 134 MB, not needed in the zip
    return model

## 2. Seed-42 results from Part 3
If you uploaded results3.zip, the seed-42 CNN and Hybrid rows are added from it so they don't need retraining.

In [ ]:
need42 = []
for name, f in [('Baseline_CNN_v2', 'table_Baseline_CNN_v2_per_class.csv'), ('Hybrid_CNN_Transformer_v2', 'table_Hybrid_CNN_Transformer_v2_per_class.csv')]:
    comp, pc = 'results3_old/table_final_comparison_v2.csv', f'results3_old/{f}'
    if os.path.exists(comp) and os.path.exists(pc):
        r = pd.read_csv(comp).set_index('model').loc[name].to_dict(); r.update({'model': name, 'seed': 42})
        p = pd.read_csv(pc, index_col=0)
        r.update({f'F1_{c}': round(p.loc[c, 'f1-score'], 4) for c in CLASSES})
        log_row(r)
        print('reused seed 42:', name)
    else:
        need42.append(name)
print('to train for seed 42:', need42 or 'none')

## 3. Train everything
Order: CNN and Hybrid seeds first, then ViT, then ResNet-18 (the slowest, about 40 minutes).

In [ ]:
B = {'Baseline_CNN_v2': build_cnn, 'Hybrid_CNN_Transformer_v2': build_hybrid}
jobs = [(B[n], 42) for n in need42] + [(build_cnn, 43), (build_hybrid, 43), (build_cnn, 44), (build_hybrid, 44),
                                       (build_vit, 42), (build_resnet18, 42)]
hybrid_demo = None
for builder, seed in jobs:
    m = run(builder, seed)
    if m.name.startswith('Hybrid'): hybrid_demo = m

In [ ]:
# PARTIAL DOWNLOAD: run this cell any time (e.g. if Colab warns it will disconnect) to save what has finished
shutil.make_archive('results4_partial', 'zip', 'results4'); files.download('results4_partial.zip')

## 4. Final paper table (mean ± std over 3 seeds for CNN and Hybrid)

In [ ]:
runs = pd.read_csv(LOG)
order = ['ResNet18_v2', 'ViT_v2', 'Baseline_CNN_v2', 'Hybrid_CNN_Transformer_v2']
pm = lambda s: f'{s.mean():.4f} ± {s.std(ddof=0):.4f}' if len(s) > 1 else f'{s.mean():.4f}'
rows = []
for n in order:
    g = runs[runs.model == n]
    if g.empty: continue
    rows.append({'Model': n.replace('_v2', '').replace('_', ' '), 'Runs': len(g), 'Params (M)': round(g.params.iloc[0] / 1e6, 2),
                 'Size (MB)': g.size_MB.iloc[0] if g.size_MB.notna().any() else None,
                 'Accuracy': pm(g.test_accuracy), 'Macro F1': pm(g.macro_F1),
                 'Inference (ms/wafer)': round(g.inference_ms_per_wafer.mean(), 2)})
final = pd.DataFrame(rows); final.to_csv('results4/table_paper_final.csv', index=False)
final

In [ ]:
f1cols = [f'F1_{c}' for c in CLASSES]
pcf = runs.groupby('model')[f1cols].mean().reindex([o for o in order if o in runs.model.values]).T
pcf.index = CLASSES; pcf.columns = [c.replace('_v2', '') for c in pcf.columns]
pcf.round(4).to_csv('results4/table_paper_per_class_f1.csv')
ax = pcf.plot.bar(figsize=(12, 4.2), width=0.8)
ax.set_ylim(0.5, 1.0); ax.set_ylabel('F1-score (mean of runs)'); ax.set_title('Per-class F1-score, all models with the same training recipe')
plt.xticks(rotation=0); plt.legend(loc='lower right', ncol=4, fontsize=8); plt.tight_layout()
plt.savefig('results4/fig_per_class_f1_final.png', dpi=300); plt.show()
pcf.round(3)

## 5. Download results

In [ ]:
shutil.make_archive('results4', 'zip', 'results4'); files.download('results4.zip')

## 6. Optional: Gradio demo (for your presentation or viva)
Run this cell and open the public link it prints. Pick a defect type to see a real test wafer, the model's prediction and its Grad-CAM heatmap. The link works for 72 hours, or until you close Colab.

In [ ]:
!pip -q install gradio
import gradio as gr
demo_model = keras.models.load_model('results3_old/Hybrid_CNN_Transformer_v2.keras') if os.path.exists('results3_old/Hybrid_CNN_Transformer_v2.keras') else hybrid_demo
gm = keras.Model(demo_model.inputs, [demo_model.get_layer('last_conv').output, demo_model.output])

def predict(defect):
    i = int(np.random.choice(np.where(y_te == CLASSES.index(defect))[0]))
    x = tf.convert_to_tensor(X_te[i][None])
    with tf.GradientTape() as tape:
        conv, pred = gm(x); c = int(tf.argmax(pred[0])); score = pred[:, c]
    g = tape.gradient(score, conv); w = tf.reduce_mean(g, axis=(0, 1, 2))
    cam = tf.nn.relu(tf.reduce_sum(conv[0] * w, -1)).numpy(); cam /= cam.max() + 1e-8
    cam = tf.image.resize(cam[..., None], (IMG, IMG)).numpy()[..., 0]
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    ax[0].imshow(X_te[i].argmax(-1), cmap='viridis'); ax[0].set_title(f'Wafer (true: {defect})'); ax[0].axis('off')
    ax[1].imshow(X_te[i].argmax(-1), cmap='gray'); ax[1].imshow(cam, cmap='jet', alpha=0.5)
    ax[1].set_title(f'Grad-CAM (pred: {CLASSES[c]})'); ax[1].axis('off'); plt.tight_layout()
    return fig, {CLASSES[k]: float(pred[0, k]) for k in range(NC)}

gr.Interface(fn=predict, inputs=gr.Dropdown(CLASSES, value='Scratch', label='Pick a defect type'),
             outputs=[gr.Plot(label='Wafer and Grad-CAM'), gr.Label(num_top_classes=3, label='Prediction')],
             title='Explainable Hybrid CNN-Transformer: Wafer Defect Classifier',
             flagging_mode='never').launch(share=True)